One may define a vector $\mathbf{x}$ of length $K$ such that $x_k = f(k)$. More generally, a tensor $\mathbf{T}$ may have values described by $t_\mathbf{k} = f(\mathbf{k})$, where $\mathbf{k}$ represents a vector of indices. However, $k_d$ must take an integer value.

A continuous tensor, or *contensor*, allows for continuous definitions. A contensor $\mathbf{T}$ may be defined such that $t_\mathbf{k} = f(\mathbf{k})$ for any real values $k_d$ within the range of its dimensions.

To show a basic operation on contensors, we recall the formula for matrix_vector multiplication:

\begin{equation}
Ax = \sum_{k=1}^K x_k A_{*,k}
\end{equation}

where the asterisk denotes all integer values within range. In other words, the multiplication is a sum of columns in the matrix, weighted by the values of the vector. The equivalent formula for 2d and 1d contensors is:

\begin{equation}
Ax = \int_{0}^K x_k A_{*,k} dk
\end{equation}

\begin{equation}
Ax = \int_{0}^K f_x(k) f_A(*,k) dk
\end{equation}

(Note: with contensors, indexing starts from zero, since it must cover all positive reals below $K$, rather than $K$ consecutive integers).

Here, the asterisk denotes all *real* values within range. Since $f_A(*,k)$ is itself a contensor, the entire integral will evaluate to one.

The following code gives definitions of addition and scalar multiplication of continuous tensors; integrals can only be approximated, so are here ommitted.

In [ ]:
class ConTensor():
  def __init__(self, rule, N):
    self.rule = rule
    self.dims = N

  def __getitem__(self, index):
    valid = True
    for k, index_k in enumerate(index):
      if index_k > self.dims[k]:
        valid = False
    if valid:
      return self.rule(index)
    raise IndexError(f"Index {index} out of bounds")

  def __setitem__(self, index, new_val):
    raise TypeError("Continuous tensor (ConTensor) object does not support item assignment")

  def __add__(self, other):
    if not isinstance(other, ConTensor):
      raise ValueError(f"Cannot add ConTensor to object of type {type(other)}")
    if self.dims != other.dims:
      raise ValueError(f"Dimensional mismatch between continuous tensors with dimension {self.dims} and {other.dims}")
    def composite_rule(index_new):
      return self.rule(index_new) + other.rule(index_new)
    return ConTensor(rule = composite_rule, N = self.dims)

  def __mul__(self, other):
    '''Performs elementwise or scalar multiplication'''
    if isinstance(other, ConTensor):
      def composite_rule(index_new):
        return self.rule(index_new) * other.rule(index_new)
      return ConTensor(rule = composite_rule, N = self.dims)
    if isinstance(other, float) or isinstance(other, int):
      def composite_rule(index_new):
        return self.rule(index_new) * other
      return ConTensor(rule = composite_rule, N = self.dims)
    raise ValueError(f"Cannot multiply ConTensor by object of type {type(other)}")

  def __rmul__(self, other):
    '''Performs scalar multiplication with operands reversed
    Elementwise multiplication of Contensors is already handled by left-multiplication'''
    if not isinstance(other, int) and not isinstance(other, float):
      raise ValueError(f"Cannot multiply ConTensor by object of type {type(other)}")
    def composite_rule(index_new):
      return self.rule(index_new) * other
    return ConTensor(rule = composite_rule, N = self.dims)


def product(x):
  tot_prod = 1
  for x_k in x:
    tot_prod *= x_k
  return tot_prod

def sum_square(x):
  tot_sum = 0
  for x_k in x:
    tot_sum += x_k**2
  return tot_sum

contensor_1 = ConTensor(rule = product, N = (5,7))
contensor_2 = ConTensor(rule = sum_square, N = (5,7))
contensor_3 = contensor_1 + contensor_2
contensor_4 = contensor_3 + contensor_2
contensor_5 = contensor_1 * contensor_3
print(contensor_1[1,2])
print(contensor_2[1,2])
print(contensor_3[1,2])
print(contensor_4[1,2])
print(contensor_5[1,2])